# GPU-accelerated fragment searching with nvMolKit

**Note:** This is a guest post from Kevin Boyd at NVIDIA.

[nvMolKit](https://github.com/NVIDIA-BioNeMo/nvMolKit) is an open-source library
with GPU-accelerated implementations of common RDKit cheminformatics operations.
It provides RDKit-compatible Python APIs for running batched molecular operations
on NVIDIA GPUs. Installation and API details are in the
[nvMolKit documentation](https://nvidia-bionemo.github.io/nvMolKit/).

We previously demonstrated nvMolKit's fingerprinting, similarity, and clustering
functionality [here](https://greglandrum.github.io/rdkit-blog/posts/2026-02-28-nvmolkit-clustering.html).

Here we demonstrate nvMolKit's substructure-search capabilities by searching
the RDKit fragment descriptor set over molecules from Enamine REAL, checking the resulting counts against RDKit and
comparing performance.

## Installation

nvMolKit is available on conda-forge alongside RDKit and requires an NVIDIA GPU
with compute capability 7.0 or later. Both packages can be installed into the
same environment:

```bash
conda install -c conda-forge rdkit nvmolkit
```

The next cell imports the packages, confirms that CUDA is available, and records
the software versions and GPU used for the benchmark.

In [ ]:
from multiprocessing import Pool
from pathlib import Path
from time import perf_counter
import csv
import importlib.metadata
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from rdkit import Chem, RDConfig, rdBase

from nvmolkit.substructure import SubstructSearchConfig, getSubstructMatches

print("Python:", platform.python_version())
print("RDKit:", rdBase.rdkitVersion)
print("nvMolKit:", importlib.metadata.version("nvmolkit"))
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "CUDA is required for nvMolKit"
print("GPU:", torch.cuda.get_device_name(0))

## Configuration

The timing results were collected on an NVIDIA GeForce RTX 5080 with an AMD
Ryzen 9 9950X. RDKit calculates the fragment counts across 16 worker processes.
SMILES parsing and molecule serialization are not timed.

In [ ]:
# Number of molecules to process
n_mols = 1_000_000

# CPU threads or worker processes for RDKit and nvMolKit preprocessing
n_cpu_threads = 6
max_fragment_matches = 1000

smi_file = Path("/home/kboyd/data/enamine_real_10M.cxsmiles")
assert smi_file.exists(), f"Enamine REAL file not found: {smi_file}"

fragment_file = Path(RDConfig.RDDataDir) / "FragmentDescriptors.csv"

config = SubstructSearchConfig(
    preprocessingThreads=n_cpu_threads,
    maxMatches=max_fragment_matches,
    uniquify=True,
    gpuIds=[0],
)

## Loading the fragment queries

RDKit's `FragmentDescriptors.csv` contains the names, descriptions, and SMARTS
definitions used by `rdkit.Chem.Fragments`. Each descriptor is the number of unique
matches of its SMARTS pattern in a molecule.

In [ ]:
fragment_names = []
fragment_descriptions = []
fragment_queries = []
with fragment_file.open() as inf:
    for row in csv.reader(inf, delimiter="\t"):
        if len(row) < 3 or not row[0] or row[0].startswith("#"):
            continue
        fragment_names.append(row[0])
        fragment_descriptions.append(row[1].strip('"'))
        fragment_queries.append(Chem.MolFromSmarts(row[2]))

assert all(fragment_queries)
print(f"Loaded {len(fragment_queries)} RDKit fragment SMARTS")
pd.DataFrame({"name": fragment_names, "description": fragment_descriptions}).head(10)

## Loading one million Enamine REAL molecules

The input file is expected at `data/enamine_real_10M.cxsmiles`; molecules can
be obtained from the [Enamine REAL 10.4M sample](https://enamine.net/compound-collections/real-compounds/real-database-subsets).


In [ ]:
mols = []
n_parse_failures = 0

with smi_file.open() as inf:
    next(inf)
    for line in inf:
        mol = Chem.MolFromSmiles(line.split("\t", 1)[0])
        if mol is None:
            n_parse_failures += 1
            continue
        mol = Chem.RemoveAllHs(mol)
        mols.append(mol)
        if len(mols) == n_mols:
            break

assert len(mols) == n_mols, f"Expected {n_mols:,} molecules, loaded {len(mols):,}"
print(f"Loaded {len(mols):,} molecules")
print(f"Parse failures skipped: {n_parse_failures:,}")

## RDKit searches

RDKit's fragment functions count unique matches with
`len(mol.GetSubstructMatches(pattern, uniquify=True))`. This implementation uses
that operation for every fragment pattern, distributing serialized molecules in
batches across`n_cpu_threads` processes. Serialization is excluded, while worker-process
startup is included.

In [ ]:
_worker_queries = None
_worker_params = None


def _rdkit_worker_init(query_binaries, max_matches):
    global _worker_queries, _worker_params
    _worker_queries = [Chem.Mol(binary) for binary in query_binaries]
    _worker_params = Chem.SubstructMatchParameters()
    _worker_params.maxMatches = max_matches
    _worker_params.uniquify = True


def _rdkit_worker_count_batch(item):
    start, mol_binaries = item
    rows = []
    for binary in mol_binaries:
        mol = Chem.Mol(binary)
        rows.append([len(mol.GetSubstructMatches(query, _worker_params)) for query in _worker_queries])
    return start, rows


mol_binaries = [mol.ToBinary() for mol in mols]
query_binaries = [query.ToBinary() for query in fragment_queries]
rdkit_batch_size = max(1, len(mol_binaries) // (n_cpu_threads * 8))
rdkit_batches = [
    (start, mol_binaries[start:start + rdkit_batch_size])
    for start in range(0, len(mol_binaries), rdkit_batch_size)
]

t0 = perf_counter()
with Pool(
    n_cpu_threads,
    initializer=_rdkit_worker_init,
    initargs=(query_binaries, max_fragment_matches),
) as pool:
    completed_batches = dict(pool.imap_unordered(_rdkit_worker_count_batch, rdkit_batches, chunksize=1))
rdkit_counts = np.asarray([
    row for start in sorted(completed_batches) for row in completed_batches[start]
], dtype=np.int32)
rdkit_time = perf_counter() - t0
rdkit_fragment_totals = rdkit_counts.sum(axis=0)

print(f"RDKit fragment counts: {rdkit_time:.2f} s")
print(f"Total fragment occurrences: {int(rdkit_fragment_totals.sum()):,}")

## nvMolKit search

nvMolKit enumerates unique matches for all molecules and SMARTS queries in one
call. Counts are obtained directly from the result offsets. An untimed
100-molecule call initializes the CUDA context. The measured call includes
preprocessing, GPU execution, and returning the match mappings to the host.

In [ ]:
getSubstructMatches(mols[:100], fragment_queries, config)

t0 = perf_counter()
nvmolkit_matches = getSubstructMatches(mols, fragment_queries, config)
nvmolkit_search_time = perf_counter() - t0

nvmolkit_counts = np.diff(nvmolkit_matches.pair_indptr).reshape(nvmolkit_matches.shape)
nvmolkit_fragment_totals = nvmolkit_counts.sum(axis=0)
assert np.array_equal(rdkit_counts, nvmolkit_counts)

print(f"nvMolKit fragment counts: {nvmolkit_search_time:.2f} s")
print(f"Total fragment occurrences: {int(nvmolkit_fragment_totals.sum()):,}")
print(f"Speedup vs RDKit: {rdkit_time / nvmolkit_search_time:.1f}x")

## Performance and descriptor counts

The left panel compares RDKit with nvMolKit. The center panel compares the total
count for each fragment descriptor and reports agreement of the complete count
matrices. The right panel shows the fragment descriptors with the largest mean
counts per molecule.

In [ ]:
fragment_results = pd.DataFrame({
    "name": fragment_names,
    "description": fragment_descriptions,
    "total occurrences": nvmolkit_fragment_totals.astype(int),
    "molecules matching": np.count_nonzero(nvmolkit_counts, axis=0),
})
fragment_results["prevalence"] = fragment_results["molecules matching"] / n_mols
fragment_results["mean count"] = fragment_results["total occurrences"] / n_mols
fragment_results = fragment_results.sort_values("total occurrences", ascending=False).reset_index(drop=True)

timings = pd.DataFrame({
    "backend": [f"RDKit\n({n_cpu_threads} processes)", "nvMolKit\n(RTX 5080)"],
    "seconds": [rdkit_time, nvmolkit_search_time],
})

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5.5))
bars = ax1.bar(
    timings["backend"],
    timings["seconds"],
    color=["#1f77b4", "#2ca02c"],
)
ax1.set_ylabel("Descriptor calculation time (seconds)")
ax1.set_title(f"{n_mols:,} molecules, {len(fragment_queries)} fragment descriptors")
ax1.grid(axis="y", alpha=0.25)
for bar, value in zip(bars, timings["seconds"]):
    ax1.text(bar.get_x() + bar.get_width()/2, value, f"{value:.2f} s", ha="center", va="bottom")

agreement = np.mean(rdkit_counts == nvmolkit_counts)
limit = int(max(rdkit_fragment_totals.max(), nvmolkit_fragment_totals.max()))
ax2.scatter(rdkit_fragment_totals, nvmolkit_fragment_totals, s=24, alpha=0.7)
ax2.plot([0, limit], [0, limit], color="black", linestyle="--", linewidth=1)
ax2.set(
    xlabel="RDKit total count",
    ylabel="nvMolKit total count",
    title="Fragment descriptor agreement",
    xlim=(0, limit),
    ylim=(0, limit),
)
ax2.set_aspect("equal", adjustable="box")
ax2.text(0.04, 0.96, f"Exact count agreement: {agreement:.3%}", transform=ax2.transAxes, va="top")
ax2.grid(alpha=0.2)

top = fragment_results.head(15).sort_values("mean count")
ax3.barh(top["name"], top["mean count"], color="#9467bd")
ax3.set_xlabel("Mean count per molecule")
ax3.set_title("Most common RDKit fragment descriptors")
ax3.grid(axis="x", alpha=0.25)

fig.tight_layout()
plt.show()

fragment_results.head(15)